# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² Mental Health Survey Dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("\nDataset published on:", metadata.datePublished)
print("Available keywords:", metadata.keywords)
print("\nDataset personal sensitive information:", metadata.personalSensitiveInformation)


## 2. Data Overview
Review available record sets, fields, and their IDs. All references are by their `@id`. Below is the list of record sets, fields, and columns as discovered from the metadata.


In [ ]:
# Explore the available record sets in the dataset

record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if not record_sets:
    print("No record sets are listed in metadata. Attempting to discover by fetching dataset records...")

# mlcroissant will fetch available record sets from schema
record_sets_dynamic = list(dataset.record_sets())

def print_record_set_info(record_sets):
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}\n  Name: {rs.get('name', 'N/A')}\n  Description: {rs.get('description', 'N/A')}\n")

if record_sets_dynamic:
    print("Discovered record sets:")
    print_record_set_info(record_sets_dynamic)
else:
    print("No record sets discovered.")

# List fields and columns for each record set
for rs in record_sets_dynamic:
    print(f"Fields for RecordSet {rs['@id']}:")
    fields = rs.get('field', [])
    for field in fields:
        print(f"  Field @id: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
        columns = field.get('column', [])
        if columns:
            for col in columns:
                print(f"    Column @id: {col['@id']} | Name: {col.get('name', 'N/A')} | Source: {col.get('source', 'N/A')}")
    print()

# As an example, preview the first few records from the first record set
if record_sets_dynamic:
    first_rs_id = record_sets_dynamic[0]['@id']
    print(f"Preview records for record set: {first_rs_id}")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        pprint.pprint(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis.
Each record set and field is referenced directly by its `@id`.


In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets_dynamic]

for rs_id in record_set_ids:
    print(f"Extracting records from record set {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}: {df.columns.tolist()}")
    print(df.head(2), "\n")

# Choose a record set for further exploration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Sample from record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations can include removing outliers, transforming data, and grouping records by key attributes.

All entities are referenced by their `@id`s. Below, we select a numeric field from the primary record set for demonstration.

In [ ]:
# Example EDA for the primary record set
# Identify candidate numeric fields to analyze

df = dataframes[main_record_set_id]

numeric_fields = []
for rs in record_sets_dynamic:
    if rs['@id'] == main_record_set_id:
        for field in rs.get('field', []):
            # Check for numeric field types
            field_type = str(field.get('dataType', '')).lower()
            if 'integer' in field_type or 'float' in field_type or 'number' in field_type:
                numeric_fields.append(field['@id'])
        break

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    print("No numeric fields found; defaulting to 'score_gad_7' if present.")
    numeric_field_id = 'score_gad_7'

if numeric_field_id not in df.columns:
    print(f"Field '{numeric_field_id}' not found in columns. The available columns are: {df.columns.tolist()}\n")

# Proceed if the numeric field exists
if numeric_field_id in df.columns:
    # Filtering records for scores above threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    field_norm_col = f"{numeric_field_id}_normalized"
    filtered_df[field_norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, field_norm_col]].head())

    # Demonstrate grouping data by categorical field
    group_fields = [col for col in df.columns if 'education' in col.lower() or 'gender' in col.lower()]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found to perform EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, show the distribution of GAD-7 scores or their relationship to education levels or gender.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution if numeric field exists
if main_record_set_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Cannot visualize: numeric field not found.")

## 6. Conclusion
This notebook demonstrated how to load, overview, extract, analyze, and visualize the FAIR² Mental Health Survey Dataset using the `mlcroissant` library, referencing all entities by their unique `@id`. We examined distributions of mental health scores and example groupings by demographic variables. This workflow lays the foundation for reproducible FAIR AI-ready analyses in real-world health survey data.